# BanglaBlocker-Base — component ablation study

This notebook measures how much each component of the BanglaBlocker gateway contributes. It uses the **same code, data splits, hyperparameters and seeds** as the BanglaBlocker-Base run reported in the paper (`BanglaGuard_XLMR_Base_AMP_FIXED_V2_SecBangla_v2_Kaggle.ipynb`), and switches off one component at a time.

## Variants (12 × seeds 42, 43, 44 = 36 training runs)

| Variant | What is removed | Group |
|---|---|---|
| `full` | nothing (reference, re-trained here so every variant shares the same environment) | 1 |
| `no_normalizer` | security-aware normalizer, at fine-tuning and at inference | 1 |
| `no_augmentation` | B1–B4 training augmentation | 1 |
| `no_tapt` | TAPT/TLM adaptation (starts from `xlm-roberta-base`) | 1 |
| `no_attention_pooling` | attention pooling ([CLS] only) | 2 |
| `no_intent_head` | intent head: its loss and the p(intent) input to the decision head | 2 |
| `no_film_policy` | FiLM policy conditioning and the domain head that drives it | 2 |
| `no_corn` | CORN ordinal-risk head (λ = 0) | 2 |
| `no_consistency` | cross-view Jensen–Shannon consistency (λ = 0) | 3 |
| `no_supcon` | supervised contrastive loss (λ = 0) | 3 |
| `no_rationale` | rationale alignment to mE5 note embeddings | 3 |
| `backbone_only` | all of the above: XLM-R Base + [CLS] + decision head, same training loop | 3 |

Everything not listed stays identical in every variant: weighted decision cross-entropy with label smoothing, layer-wise LR decay, batch of 16 scenarios (64 texts), 12 epochs with patience 3 on validation Macro-F1, fixed head-initialisation seed, max length 128.

One design choice to note: the TAPT/TLM checkpoint is trained once and shared by every variant that uses it. It is trained on normalised text, so `no_normalizer` removes the normalizer from fine-tuning and inference only.

## Metrics saved per run

Held-out test (3,600 prompts): three-class Macro-F1, per-class F1, accuracy, detection Macro-F1, recall, specificity, AUROC, hard-negative over-refusal, decision-matched Macro-F1, and seven-class intent Macro-F1. It also records Macro-F1 on the unseen-perturbation stress set, which is what the normalizer and augmentation are meant to help. The last cells compute mean ± SD over seeds, and a family-clustered paired bootstrap (5,000 resamples) of every variant against `full`, as in the paper.

## How to run on Kaggle (T4 GPU)

About 13 minutes per run, plus about 40 minutes for TAPT/TLM once. Split into three sessions of roughly 3 hours:

1. **Session 1:** `RUN_GROUP = 1` → *Save & Run All (Commit)*. Afterwards, add this notebook's output as an input to the notebook (*Add Input → Your Work*).
2. **Session 2:** `RUN_GROUP = 2`. The attached output lets the notebook reuse the TAPT/TLM checkpoint and import the finished runs.
3. **Session 3:** `RUN_GROUP = 3`, with the outputs of sessions 1 and 2 attached.

`RUN_GROUP = "all"` runs everything in one session, about 9 hours; this fits within Kaggle's 12-hour limit. The notebook skips runs that are already finished, so an interrupted session can simply be restarted. Set `SMOKE_RUN = True` for a few-minute check of the whole pipeline on tiny data.

**Send back:** `/kaggle/working/banglablocker_ablation_results.zip`. It contains every finished run, including imported ones, plus the summary tables.


In [ ]:
# ============================================================
# CELL 1 — INSTALL + GLOBAL CONFIG
# ============================================================
!pip -q install -U "transformers>=4.50,<5" "accelerate>=1.0" \
    "sentencepiece>=0.1.99" "safetensors>=0.4" "scikit-learn>=1.3" 2>/dev/null || echo "pip install skipped/offline"

import os, re, gc, json, math, time, random, shutil, zipfile, warnings, unicodedata
from pathlib import Path
from contextlib import nullcontext
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             roc_auc_score)
from transformers import (AutoTokenizer, AutoModel, AutoModelForMaskedLM,
                          DataCollatorForLanguageModeling, get_linear_schedule_with_warmup)
warnings.filterwarnings("ignore")
try:
    display
except NameError:            # plain-python execution
    def display(x): print(x.to_string() if hasattr(x, "to_string") else x)

# ---------------- Run controls ----------------
SMOKE_RUN = False            # True = few-minute pipeline check on tiny data
RUN_GROUP = 1                # 1, 2, 3 or "all"  (see the table above)
SEEDS = [42, 43, 44]
SMOKE_SEEDS = [42]

# ---------------- Backbones ----------------
BACKBONE = os.environ.get("BB_BACKBONE", "FacebookAI/xlm-roberta-base")
RATIONALE_MODEL = os.environ.get("BB_RATIONALE_MODEL", "intfloat/multilingual-e5-large")
MODEL_SLUG = "banglablocker_base_ablation"

# ---------------- Task ----------------
TEXT_VIEWS = ["bangla", "banglish", "code_mix", "english"]
LABELS = ["ALLOW", "REVIEW", "BLOCK"]; LABEL2ID = {x: i for i, x in enumerate(LABELS)}; ID2LABEL = {i: x for x, i in LABEL2ID.items()}
DOMAINS = ["banking", "healthcare", "e-commerce"]; DOMAIN2ID = {x: i for i, x in enumerate(DOMAINS)}
INTENTS = ["B0_legitimate_security", "S1_jailbreak", "S2_prompt_extraction", "S3_data_exfiltration",
           "S4_unauthorized_action", "S5_authentication_authorization_bypass", "S6_indirect_injection"]
INTENT2ID = {x: i for i, x in enumerate(INTENTS)}
RISKS = ["none", "low", "medium", "high", "very_high"]; RISK2ID = {x: i for i, x in enumerate(RISKS)}

# ---------------- Hyperparameters: identical to the paper's BanglaBlocker-Base run ----------------
MAX_LEN = 128
EPOCHS = 12
PATIENCE = 3
LR_ENCODER = 3e-5
LR_HEADS = 3e-4
LAYERWISE_LR_DECAY = 0.90
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0
DROPOUT = 0.10
POLICY_DROPOUT = 0.20
LABEL_SMOOTHING = 0.05
SUPCON_TAU = 0.07
CLASS_W = torch.tensor([0.96, 1.30, 0.74], dtype=torch.float32)
TRAIN_SCENARIO_BATCH = 4      # 16 texts per micro-batch
GRAD_ACCUM = 4                # effective 16 scenarios = 64 texts
EVAL_SCENARIO_BATCH = 16
AUGMENT_PROB_DEFAULT = 0.30
FIXED_HEAD_INIT_SEED = 2026
GRADIENT_CHECKPOINTING = True

TAPT_EPOCHS = 3
TAPT_LR = 3e-5
TAPT_BATCH = 4
TAPT_GRAD_ACCUM = 2

# Smoke controls
SMOKE_TRAIN_N, SMOKE_VAL_N, SMOKE_TEST_N = 48, 24, 36
SMOKE_MAX_STEPS, SMOKE_TAPT_STEPS = 6, 6

N_BOOT = 200 if SMOKE_RUN else 5000

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
CC = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
AMP_DTYPE = torch.bfloat16 if (torch.cuda.is_available() and CC[0] >= 8) else torch.float16
USE_SCALER = torch.cuda.is_available() and AMP_DTYPE == torch.float16

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def free_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def amp_context():
    return torch.autocast("cuda", dtype=AMP_DTYPE) if torch.cuda.is_available() else nullcontext()

def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=USE_SCALER)

WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
OUT_DIR = WORK_ROOT / MODEL_SLUG
RUNS_DIR = OUT_DIR / "ablation_runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("GPU:", GPU_NAME, "| AMP:", AMP_DTYPE, "| SMOKE_RUN:", SMOKE_RUN, "| RUN_GROUP:", RUN_GROUP)
print("OUT_DIR:", OUT_DIR)


In [ ]:
# ============================================================
# CELL 2 — ABLATION VARIANTS
# ============================================================
BASE_CFG = dict(
    normalize=True,          # security-aware normalizer (fine-tuning + inference)
    tapt=True,               # TAPT/TLM-adapted backbone
    augment_prob=AUGMENT_PROB_DEFAULT,
    attn_pool=True,          # [CLS] || attention pooling
    intent=True,             # intent loss + p(intent) fed to the decision head
    film=True,               # FiLM policy conditioning + domain head that drives it
    lambda_int=0.50, lambda_dom=0.10, lambda_ord=0.30,
    lambda_cons=1.00, lambda_scl=0.10, lambda_rat=0.20,
    rationale=True,          # rationale alignment to frozen mE5 note embeddings
)

VARIANTS = OrderedDict([
    ("full", {}),
    ("no_normalizer", dict(normalize=False)),
    ("no_augmentation", dict(augment_prob=0.0)),
    ("no_tapt", dict(tapt=False)),
    ("no_attention_pooling", dict(attn_pool=False)),
    ("no_intent_head", dict(intent=False)),
    ("no_film_policy", dict(film=False)),
    ("no_corn", dict(lambda_ord=0.0)),
    ("no_consistency", dict(lambda_cons=0.0)),
    ("no_supcon", dict(lambda_scl=0.0)),
    ("no_rationale", dict(rationale=False)),
    ("backbone_only", dict(normalize=False, tapt=False, augment_prob=0.0, attn_pool=False,
                           intent=False, film=False, lambda_ord=0.0, lambda_cons=0.0,
                           lambda_scl=0.0, rationale=False)),
])

GROUPS = {
    1: ["full", "no_normalizer", "no_augmentation", "no_tapt"],
    2: ["no_attention_pooling", "no_intent_head", "no_film_policy", "no_corn"],
    3: ["no_consistency", "no_supcon", "no_rationale", "backbone_only"],
}
assert sorted(sum(GROUPS.values(), [])) == sorted(VARIANTS), "every variant must be in exactly one group"

def variant_cfg(name):
    cfg = dict(BASE_CFG); cfg.update(VARIANTS[name])
    if not cfg["intent"]:    cfg["lambda_int"] = 0.0
    if not cfg["film"]:      cfg["lambda_dom"] = 0.0      # the domain head only serves FiLM
    if not cfg["rationale"]: cfg["lambda_rat"] = 0.0
    cfg["name"] = name
    return cfg

if SMOKE_RUN:
    RUN_VARIANTS = list(VARIANTS)                         # exercise every switch
    RUN_SEEDS = SMOKE_SEEDS
elif RUN_GROUP == "all":
    RUN_VARIANTS = list(VARIANTS); RUN_SEEDS = SEEDS
else:
    RUN_VARIANTS = GROUPS[int(RUN_GROUP)]; RUN_SEEDS = SEEDS

NEED_TAPT = any(variant_cfg(v)["tapt"] for v in RUN_VARIANTS)
NEED_RATIONALE = any(variant_cfg(v)["rationale"] for v in RUN_VARIANTS)

print(pd.DataFrame([{k: variant_cfg(v)[k] for k in ["name"] + list(BASE_CFG)} for v in VARIANTS]).to_string(index=False))
print("\nthis session runs:", RUN_VARIANTS, "x seeds", RUN_SEEDS)


In [ ]:
# ============================================================
# CELL 3 — RESOLVE + AUDIT splits_v2  (same checks as the paper run)
# ============================================================
MASTER_FILENAME = "SecBangla_clean_master_with_splits.csv"

def _valid_dir(p):
    p = Path(p); return p.is_dir() and (p / MASTER_FILENAME).exists() and (p / "train.csv").exists()

def resolve_data_dir():
    for p in [Path("./splits_v2"), Path("/mnt/data/splits_v2")]:
        if _valid_dir(p): return p
    root = Path("/kaggle/input")
    if root.exists():
        for f in root.rglob(MASTER_FILENAME):
            if _valid_dir(f.parent): return f.parent
        for z in root.rglob("*.zip"):
            try:
                with zipfile.ZipFile(z, "r") as zf:
                    if not any(n.endswith(MASTER_FILENAME) for n in zf.namelist()): continue
                    dst = WORK_ROOT / "_splits_v2_extract"; dst.mkdir(parents=True, exist_ok=True); zf.extractall(dst)
                    for f in dst.rglob(MASTER_FILENAME):
                        if _valid_dir(f.parent): return f.parent
            except zipfile.BadZipFile:
                pass
    raise FileNotFoundError("Attach the extracted splits_v2 dataset or splits_v2.zip.")

DATA_DIR = resolve_data_dir()
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "heldout_test.csv")

REQUIRED = ["id", "domain", "security_intent", "hard_negative_type", "decision", "risk_level",
            "bangla", "banglish", "code_mix", "english", "note", "scenario_family_id", "split",
            "in_decision_matched_test"]
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    miss = [c for c in REQUIRED if c not in df.columns]; assert not miss, f"{name} missing {miss}"
    assert df.id.is_unique
assert set(train_df.scenario_family_id).isdisjoint(val_df.scenario_family_id)
assert set(train_df.scenario_family_id).isdisjoint(test_df.scenario_family_id)
assert set(val_df.scenario_family_id).isdisjoint(test_df.scenario_family_id)
for v in TEXT_VIEWS:
    assert set(train_df[v]).isdisjoint(val_df[v]); assert set(train_df[v]).isdisjoint(test_df[v])
print("DATA_DIR:", DATA_DIR, {"train": len(train_df), "val": len(val_df), "test": len(test_df)})
print("Leakage audit: PASS")

def smoke_sample(df, n, seed):
    if len(df) <= n: return df.copy().reset_index(drop=True)
    tmp = df.copy(); tmp["_s"] = tmp.domain.astype(str) + "|" + tmp.decision.astype(str)
    rng = np.random.RandomState(seed); groups = list(tmp.groupby("_s", sort=False)); per = max(1, n // len(groups))
    parts = [g.sample(min(per, len(g)), random_state=int(rng.randint(2**31 - 1))) for _, g in groups]
    out = pd.concat(parts).drop(columns="_s").drop_duplicates("id")
    if len(out) < n:
        rem = df[~df.id.isin(out.id)]; out = pd.concat([out, rem.sample(min(n - len(out), len(rem)), random_state=seed)])
    return out.sample(min(n, len(out)), random_state=seed).reset_index(drop=True)

if SMOKE_RUN:
    train_df = smoke_sample(train_df, SMOKE_TRAIN_N, 42)
    val_df = smoke_sample(val_df, SMOKE_VAL_N, 43)
    test_df = smoke_sample(test_df, SMOKE_TEST_N, 44)
    print("SMOKE counts:", len(train_df), len(val_df), len(test_df))


In [ ]:
# ============================================================
# CELL 4 — NORMALIZER + B1–B4 AUGMENTATION  (verbatim from the paper run)
# ============================================================
_ZW = re.compile("[​⁠﻿]")  # keep U+200C/U+200D (needed for Bengali conjuncts)
_HOMO = str.maketrans({"а": "a", "е": "e", "о": "o", "р": "p", "с": "c", "х": "x", "і": "i", "ѕ": "s",
                       "ο": "o", "α": "a", "ε": "e", "ρ": "p"})
_ASCII_TO_HOMO = {"a": "а", "e": "е", "o": "о", "p": "р", "c": "с", "x": "х", "i": "і", "s": "ѕ"}

def normalize_security_text(text):
    t = unicodedata.normalize("NFKC", str(text))
    t = _ZW.sub("", t).translate(_HOMO)
    frag = re.compile(r"(?i)\b(?:[a-z][\s\-._*]){3,}[a-z]\b")
    t = frag.sub(lambda m: re.sub(r"[\s\-._*]", "", m.group(0)), t)
    return re.sub(r"\s+", " ", t).strip()

def identity_text(text):
    return str(text)

BANGLISH_RULES = [
    ("sh","s"),("s","sh"),("chh","ch"),("ch","chh"),("bh","v"),("v","bh"),("ph","f"),("f","ph"),("kh","k"),("gh","g"),
    ("th","t"),("dh","d"),("aa","a"),("ee","i"),("ii","i"),("oo","u"),("ou","o"),("au","o"),("achhe","ache"),("ache","ase"),
    ("ase","ache"),("nai","nei"),("nei","nai"),("koro","kro"),("kro","koro"),("koru","koro"),("hobe","hbe"),("hbe","hobe"),
    ("amar","amr"),("tomar","tmr"),("apnar","apnr"),("account","acount"),("password","passwrd"),("verify","verfy"),
    ("verification","verfication"),("please","pls"),("plz","pls"),("message","msg"),("number","num"),
    ("transaction","transction"),("payment","paymnt"),("transfer","transfr"),("security","securty"),("system","systm"),
    ("prompt","promt"),("instruction","instrction"),("kivabe","kibhabe"),("kibhabe","kivabe"),("ki","kii"),("ami","amii"),
    ("taka","tk"),("ekhon","ekhn"),("lagbe","lgbe"),("dorkar","drkar"),("korte","krte"),("korbo","krbo"),("bolo","blo"),
    ("dao","daw"),("diye","dia"),("jabe","jbe"),("parbo","prbo"),("chai","chy"),
]
LOANWORD_FLIPS = {
    "debit card":"ডেবিট কার্ড","credit card":"ক্রেডিট কার্ড","account":"অ্যাকাউন্ট","otp":"ওটিপি","password":"পাসওয়ার্ড",
    "refund":"রিফান্ড","invoice":"ইনভয়েস","payment":"পেমেন্ট","transaction":"ট্রানজ্যাকশন","transfer":"ট্রান্সফার",
    "security":"সিকিউরিটি","system":"সিস্টেম","prompt":"প্রম্পট","login":"লগইন","verification":"ভেরিফিকেশন",
    "email":"ইমেইল","sms":"এসএমএস","api":"এপিআই",
}
CARRIERS = ["SMS content: {x}", "Email body: {x}", "Invoice note: {x}", "Résumé text: {x}",
            "JSON field `message`: {x}", "Web snippet: {x}", "Vendor memo: {x}"]

def _obfuscate(t, rng):
    toks = t.split()
    if not toks: return t
    mode = rng.choice(["fragment", "spacing", "zero_width", "homoglyph", "punct", "repeat", "hasanta_drop"])
    j = rng.randrange(len(toks)); tok = toks[j]
    if mode == "fragment" and tok.isascii() and tok.isalpha() and len(tok) >= 5: toks[j] = "-".join(tok)
    elif mode == "spacing" and tok.isascii() and tok.isalpha() and len(tok) >= 5: toks[j] = " ".join(tok)
    elif mode == "zero_width" and len(tok) >= 4:
        k = max(1, len(tok) // 2); toks[j] = tok[:k] + "​" + tok[k:]
    elif mode == "homoglyph" and tok.isascii():
        chars = list(tok); cand = [i for i, c in enumerate(chars) if c.lower() in _ASCII_TO_HOMO]
        if cand:
            k = rng.choice(cand); chars[k] = _ASCII_TO_HOMO[chars[k].lower()]; toks[j] = "".join(chars)
    elif mode == "punct" and len(tok) >= 4: toks[j] = tok + "..."
    elif mode == "repeat" and len(tok) >= 3:
        k = rng.randrange(len(tok)); toks[j] = tok[:k] + tok[k] * 2 + tok[k:]
    elif mode == "hasanta_drop" and "্" in tok: toks[j] = tok.replace("্", "", 1)
    return " ".join(toks)

def augment_text(text, view, intent, rng):
    original = str(text); t = original
    if view in ("banglish", "code_mix"):                                   # B1
        for _ in range(rng.randint(1, 3)):
            a, b = rng.choice(BANGLISH_RULES); t = re.sub(re.escape(a), b, t, count=1, flags=re.I)
    if view in ("bangla", "code_mix"):                                     # B2
        flip_p = rng.uniform(0.20, 0.50)
        for a, b in LOANWORD_FLIPS.items():
            if rng.random() < flip_p: t = re.sub(re.escape(a), b, t, flags=re.I)
    t = _obfuscate(t, rng)                                                 # B3
    if intent in ("S6_indirect_injection", "B0_legitimate_security") and rng.random() < 0.50:   # B4
        t = rng.choice(CARRIERS).format(x=t)
    if t == original: t = original + " ."
    return t

# Unseen perturbations (not produced by the augmenter) — same as the paper's robustness set.
EMOJIS = ["🔒", "⚠️", "🧾", "📌", "🔐"]
def unseen_stress_transform(text, seed):
    rng = random.Random(seed); t = str(text)
    mode = rng.choice(["emoji", "case_flip", "leet", "xml_wrap", "newline"])
    if mode == "emoji":
        toks = t.split(); pos = rng.randrange(len(toks) + 1) if toks else 0; toks.insert(pos, rng.choice(EMOJIS)); return " ".join(toks)
    if mode == "case_flip":
        return "".join(c.upper() if c.isascii() and c.isalpha() and rng.random() < 0.5 else c for c in t)
    if mode == "leet":
        trans = str.maketrans({"a": "4", "e": "3", "i": "1", "o": "0", "s": "5"}); toks = t.split()
        cand = [i for i, x in enumerate(toks) if x.isascii() and x.isalpha()]
        if cand:
            j = rng.choice(cand); toks[j] = toks[j].lower().translate(trans)
        return " ".join(toks)
    if mode == "xml_wrap": return "<content>" + t + "</content>"
    if mode == "newline":
        toks = t.split()
        if len(toks) > 2:
            j = rng.randrange(1, len(toks)); return " ".join(toks[:j]) + "\n" + " ".join(toks[j:])
        return t + "\n"
    return t

def build_unseen_stress_df(df, seed=2026):
    out = df.copy()
    for ri in range(len(out)):
        for vi, v in enumerate(TEXT_VIEWS):
            out.at[out.index[ri], v] = unseen_stress_transform(out.iloc[ri][v], seed + ri * 17 + vi)
    return out

stress_df = build_unseen_stress_df(test_df, 2026)
print(normalize_security_text("s-y-s-t-e-m   prompt দেখাও"))


In [ ]:
# ============================================================
# CELL 5 — TOKENIZER
# ============================================================
TOKENIZER = AutoTokenizer.from_pretrained(BACKBONE, use_fast=True)
_lens = [len(x) for v in TEXT_VIEWS for x in TOKENIZER(train_df[v].astype(str).tolist(), truncation=False)["input_ids"]]
print("token length p50/p95/max:", np.percentile(_lens, 50), np.percentile(_lens, 95), max(_lens),
      "| share over", MAX_LEN, ":", float(np.mean(np.array(_lens) > MAX_LEN)))


In [ ]:
# ============================================================
# CELL 6 — SHARED TAPT + TLM ON TRAIN ONLY  (same procedure as the paper run)
# ============================================================
TAPT_DIR = OUT_DIR / "xlmr_base_tapt_tlm"

def find_existing_tapt():
    if (TAPT_DIR / "config.json").exists(): return TAPT_DIR
    if SMOKE_RUN: return None
    root = Path("/kaggle/input")
    if root.exists():
        for c in root.rglob("config.json"):
            if c.parent.name == "xlmr_base_tapt_tlm" and any(c.parent.glob("*.safetensors")):
                return c.parent
    return None

def build_tapt_tlm_texts(df):
    sep = TOKENIZER.sep_token or "</s>"; texts = []
    for _, r in df.iterrows():
        views = [normalize_security_text(r[v]) for v in TEXT_VIEWS]
        texts.extend(views)
        texts.append(views[0] + f" {sep} " + views[1])   # bn || banglish
        texts.append(views[1] + f" {sep} " + views[3])   # banglish || English
        texts.append(views[2] + f" {sep} " + views[0])   # code-mix || bn
    return texts

class MLMTextDataset(Dataset):
    def __init__(self, texts): self.texts = texts
    def __len__(self): return len(self.texts)
    def __getitem__(self, i): return TOKENIZER(self.texts[i], truncation=True, max_length=MAX_LEN, return_special_tokens_mask=True)

def run_tapt():
    print("Running TAPT/TLM from", BACKBONE); t0 = time.time()
    mlm = AutoModelForMaskedLM.from_pretrained(BACKBONE).to(DEVICE)
    if GRADIENT_CHECKPOINTING and hasattr(mlm, "gradient_checkpointing_enable"): mlm.gradient_checkpointing_enable()
    texts = build_tapt_tlm_texts(train_df)
    if SMOKE_RUN: texts = texts[:96]
    gen = torch.Generator(); gen.manual_seed(2026)
    dl = DataLoader(MLMTextDataset(texts), batch_size=TAPT_BATCH, shuffle=True,
                    collate_fn=DataCollatorForLanguageModeling(TOKENIZER, mlm_probability=0.15), generator=gen)
    opt = torch.optim.AdamW(mlm.parameters(), lr=TAPT_LR, weight_decay=WEIGHT_DECAY); scaler = make_scaler()
    epochs = 1 if SMOKE_RUN else TAPT_EPOCHS; max_steps = SMOKE_TAPT_STEPS if SMOKE_RUN else None
    total = max_steps if max_steps else math.ceil(len(dl) / TAPT_GRAD_ACCUM) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP_RATIO * total), total)
    step = 0; opt.zero_grad(set_to_none=True)
    for ep in range(epochs):
        mlm.train()
        for bi, batch in enumerate(dl):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with amp_context(): loss = mlm(**batch).loss / TAPT_GRAD_ACCUM
            scaler.scale(loss).backward()
            if (bi + 1) % TAPT_GRAD_ACCUM == 0 or (bi + 1) == len(dl):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(mlm.parameters(), MAX_GRAD_NORM)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
                if step % 200 == 0: print("TAPT/TLM", {"epoch": ep + 1, "update": step, "loss": float(loss.detach()) * TAPT_GRAD_ACCUM})
                if max_steps and step >= max_steps: break
        if max_steps and step >= max_steps: break
    mlm.save_pretrained(TAPT_DIR, safe_serialization=True); TOKENIZER.save_pretrained(TAPT_DIR)
    del mlm; free_memory()
    print(f"TAPT/TLM done in {(time.time() - t0) / 60:.1f} min")
    return TAPT_DIR

TAPT_BACKBONE = None
if NEED_TAPT:
    found = find_existing_tapt()
    if found is not None:
        print("Reusing TAPT/TLM checkpoint:", found); TAPT_BACKBONE = str(found)
    else:
        TAPT_BACKBONE = str(run_tapt())
print("TAPT backbone:", TAPT_BACKBONE, "| plain backbone:", BACKBONE)


In [ ]:
# ============================================================
# CELL 7 — FROZEN mE5 RATIONALE EMBEDDINGS (train only; training-time supervision only)
# ============================================================
def mean_pool(H, mask):
    m = mask.unsqueeze(-1).to(H.dtype); return (H * m).sum(1) / m.sum(1).clamp_min(1)

def build_note_embeddings(df):
    cache = OUT_DIR / ("note_emb_smoke.npz" if SMOKE_RUN else "note_emb_train.npz")
    unique = df[["id", "note"]].drop_duplicates("id").copy(); unique["note"] = unique["note"].fillna("").astype(str)
    if cache.exists():
        x = np.load(cache, allow_pickle=True); ids = x["ids"].tolist(); emb = x["emb"]
        if set(map(str, ids)) >= set(unique.id.astype(str)):
            return {str(k): emb[i] for i, k in enumerate(ids)}, emb.shape[1]
    print("Encoding train-only notes with", RATIONALE_MODEL)
    tok = AutoTokenizer.from_pretrained(RATIONALE_MODEL)
    enc = AutoModel.from_pretrained(RATIONALE_MODEL, torch_dtype=AMP_DTYPE if torch.cuda.is_available() else torch.float32).to(DEVICE).eval()
    vecs = []; notes = ["passage: " + x for x in unique.note.tolist()]; bs = 16
    for i in range(0, len(notes), bs):
        b = tok(notes[i:i + bs], padding=True, truncation=True, max_length=128, return_tensors="pt")
        b = {k: v.to(DEVICE) for k, v in b.items()}
        with torch.no_grad(), amp_context():
            H = enc(**b).last_hidden_state; z = F.normalize(mean_pool(H, b["attention_mask"]).float(), dim=-1)
        vecs.append(z.cpu().numpy())
    emb = np.concatenate(vecs, 0).astype("float32")
    np.savez_compressed(cache, ids=unique.id.astype(str).values, emb=emb)
    del enc, tok; free_memory()
    return {str(k): emb[i] for i, k in enumerate(unique.id.astype(str).tolist())}, emb.shape[1]

NOTE_EMB, RAT_DIM = build_note_embeddings(train_df) if NEED_RATIONALE else (None, 0)
print("RAT_DIM:", RAT_DIM)


In [ ]:
# ============================================================
# CELL 8 — SCENARIO-MAJOR 4-VIEW DATASET (per-variant normalizer / augmentation)
# ============================================================
class ScenarioDataset(Dataset):
    def __init__(self, df, cfg, augment=False, seed=42):
        self.df = df.reset_index(drop=True); self.cfg = cfg; self.augment = augment; self.seed = seed
        self.norm = normalize_security_text if cfg["normalize"] else identity_text
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]; rng = random.Random(self.seed * 1000003 + i); texts = []
        for v in TEXT_VIEWS:
            t = self.norm(r[v])
            if self.augment and self.cfg["augment_prob"] > 0 and rng.random() < self.cfg["augment_prob"]:
                t = augment_text(t, v, r.security_intent, rng)
            texts.append(t)
        return {"texts": texts, "id": str(r.id), "decision": LABEL2ID[r.decision], "intent": INTENT2ID[r.security_intent],
                "domain": DOMAIN2ID[r.domain], "risk": RISK2ID[r.risk_level]}

def make_collate(cfg, training_aux=False):
    def collate(batch):
        texts = [t for x in batch for t in x["texts"]]
        enc = TOKENIZER(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
        B = len(batch)
        for k in ["decision", "intent", "domain", "risk"]:
            enc[k] = torch.tensor([x[k] for x in batch], dtype=torch.long).repeat_interleave(4)
        enc["scenario"] = torch.arange(B, dtype=torch.long).repeat_interleave(4)
        enc["ids"] = [x["id"] for x in batch]
        if training_aux and cfg["rationale"]:
            ne = np.stack([NOTE_EMB[str(x["id"])] for x in batch])
            enc["note_emb"] = torch.tensor(ne, dtype=torch.float32).repeat_interleave(4, dim=0)
        return enc
    return collate


In [ ]:
# ============================================================
# CELL 9 — MODEL + LOSSES WITH ABLATION SWITCHES
#   With every switch on, this is exactly the paper's BanglaBlocker-Base.
# ============================================================
class BanglaBlockerAblation(nn.Module):
    def __init__(self, backbone, cfg, n_dom=3, n_int=7, n_dec=3, n_risk=5, pol_dim=64, rat_dim=1024, proj_dim=128, dropout=0.1):
        super().__init__()
        self.cfg = cfg
        self.enc = AutoModel.from_pretrained(backbone, add_pooling_layer=False)
        d = self.enc.config.hidden_size; self.n_dom = n_dom
        if cfg["attn_pool"]:
            self.attn = nn.Linear(d, 1)
            self.pool = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.LayerNorm(d))
        else:
            self.attn = None
            self.pool = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.LayerNorm(d))
        self.drop = nn.Dropout(dropout)
        self.dom_head = nn.Linear(d, n_dom)
        self.int_head = nn.Linear(d, n_int)
        self.risk_head = nn.Linear(d, n_risk - 1)                     # CORN
        if cfg["film"]:
            self.policy = nn.Embedding(n_dom + 1, pol_dim)             # last index = generic policy
            self.film = nn.Linear(pol_dim, 2 * d); nn.init.zeros_(self.film.weight); nn.init.zeros_(self.film.bias)
        else:
            self.policy = None; self.film = None
        dec_in = d + (n_int if cfg["intent"] else 0)
        self.dec_head = nn.Sequential(nn.Linear(dec_in, d), nn.GELU(), nn.Dropout(dropout), nn.Linear(d, n_dec))
        self.proj = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, proj_dim))
        self.rat = nn.Linear(d, rat_dim) if (cfg["rationale"] and rat_dim > 0) else None

    def forward(self, input_ids, attention_mask, policy_dropout=0.0):
        H = self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        H_pool = H.float(); cls = H_pool[:, 0]
        if self.attn is not None:
            a = self.attn(H).squeeze(-1).float().masked_fill(attention_mask == 0, -1e4)   # AMP-safe (T4)
            att = (a.softmax(-1).unsqueeze(-1) * H_pool).sum(1)
            h = self.drop(self.pool(torch.cat([cls, att], -1)))
        else:
            h = self.drop(self.pool(cls))
        dom_logits = self.dom_head(h); int_logits = self.int_head(h); risk_logits = self.risk_head(h)
        if self.film is not None:
            p_dom = dom_logits.detach().softmax(-1)
            w = torch.cat([p_dom, p_dom.new_zeros(p_dom.size(0), 1)], -1)
            if self.training and policy_dropout > 0:
                generic = F.one_hot(torch.tensor(self.n_dom, device=h.device), self.n_dom + 1).to(h.dtype)
                mask = torch.rand(h.size(0), 1, device=h.device) < policy_dropout
                w = torch.where(mask, generic.expand_as(w), w)
            gamma, beta = self.film(w @ self.policy.weight.to(h.dtype)).chunk(2, -1)
            h_pol = h * (1 + gamma) + beta
        else:
            h_pol = h
        dec_in = torch.cat([h_pol, int_logits.softmax(-1)], -1) if self.cfg["intent"] else h_pol
        return {"h": h, "dec": self.dec_head(dec_in), "int": int_logits, "dom": dom_logits, "risk": risk_logits}

def corn_loss(logits, y):
    losses = []; count = 0
    for k in range(logits.size(1)):
        m = y >= k
        if m.any():
            losses.append(F.binary_cross_entropy_with_logits(logits[m, k], (y[m] > k).float(), reduction="sum")); count += int(m.sum())
    return torch.stack(losses).sum() / max(count, 1)

def multiview_js(logits, n_views=4):
    ctx = torch.autocast("cuda", enabled=False) if logits.is_cuda else nullcontext()
    with ctx:
        x = logits.float(); logp = x.log_softmax(-1).view(-1, n_views, x.size(-1)); p = logp.exp()
        logm = p.mean(1, keepdim=True).clamp_min(1e-8).log()
        return (p * (logp - logm)).sum(-1).mean()

def supcon(z, group, tau=0.07):
    ctx = torch.autocast("cuda", enabled=False) if z.is_cuda else nullcontext()
    with ctx:
        zf = F.normalize(z.float(), dim=-1); gf = group.to(zf.device)
        sim = (zf @ zf.T) / float(tau); eye = torch.eye(len(zf), dtype=torch.bool, device=zf.device)
        sim = sim.masked_fill(eye, -1e4); pos = (gf[:, None] == gf[None, :]) & ~eye
        denom = pos.sum(1); valid = denom > 0
        if not valid.any(): return zf.new_tensor(0.0)
        logp = sim - sim.logsumexp(1, keepdim=True)
        return -(logp[valid] * pos[valid]).sum(1).div(denom[valid]).mean()

def total_loss(out, batch, model, cfg):
    ce = F.cross_entropy(out["dec"].float(), batch["decision"], weight=CLASS_W.to(out["dec"].device), label_smoothing=LABEL_SMOOTHING)
    loss = ce; pieces = {"decision_ce": float(ce.detach())}
    if cfg["lambda_int"] > 0:
        li = F.cross_entropy(out["int"].float(), batch["intent"], label_smoothing=LABEL_SMOOTHING)
        loss = loss + cfg["lambda_int"] * li; pieces["intent"] = float(li.detach())
    if cfg["lambda_dom"] > 0:
        ld = F.cross_entropy(out["dom"].float(), batch["domain"])
        loss = loss + cfg["lambda_dom"] * ld; pieces["domain"] = float(ld.detach())
    if cfg["lambda_ord"] > 0:
        lo = corn_loss(out["risk"].float(), batch["risk"])
        loss = loss + cfg["lambda_ord"] * lo; pieces["corn"] = float(lo.detach())
    if cfg["lambda_cons"] > 0:
        lc = multiview_js(out["dec"]) + (0.5 * multiview_js(out["int"]) if cfg["intent"] else 0.0)
        loss = loss + cfg["lambda_cons"] * lc; pieces["consistency"] = float(lc.detach())
    if cfg["lambda_scl"] > 0:
        ls = supcon(model.proj(out["h"]), batch["scenario"], SUPCON_TAU)
        loss = loss + cfg["lambda_scl"] * ls; pieces["supcon"] = float(ls.detach())
    if cfg["lambda_rat"] > 0 and model.rat is not None:
        lr = (1 - F.cosine_similarity(model.rat(out["h"]).float(), batch["note_emb"], dim=-1)).mean()
        loss = loss + cfg["lambda_rat"] * lr; pieces["rationale"] = float(lr.detach())
    return loss, pieces

# AMP regression test (the T4 failure mode fixed in the paper run)
_z = torch.randn(16, 128, device=DEVICE, dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
assert torch.isfinite(supcon(_z, torch.arange(4, device=DEVICE).repeat_interleave(4), 0.07)); del _z
print("Model/loss definitions ready. AMP SupCon regression test: PASS")


In [ ]:
# ============================================================
# CELL 10 — SWITCH VERIFICATION (runs before any training)
#   Builds each architecture variant on a probe batch and checks that the
#   ablated module/loss is really absent and everything else is present.
# ============================================================
def backbone_for(cfg):
    return TAPT_BACKBONE if cfg["tapt"] else BACKBONE

def build_model(cfg):
    seed_all(FIXED_HEAD_INIT_SEED)
    m = BanglaBlockerAblation(backbone_for(cfg), cfg, rat_dim=RAT_DIM if cfg["rationale"] else 0, dropout=DROPOUT)
    if GRADIENT_CHECKPOINTING and hasattr(m.enc, "gradient_checkpointing_enable"): m.enc.gradient_checkpointing_enable()
    return m.to(DEVICE)

EXPECTED_PIECES = {"decision_ce": lambda c: True, "intent": lambda c: c["lambda_int"] > 0, "domain": lambda c: c["lambda_dom"] > 0,
                   "corn": lambda c: c["lambda_ord"] > 0, "consistency": lambda c: c["lambda_cons"] > 0,
                   "supcon": lambda c: c["lambda_scl"] > 0, "rationale": lambda c: c["lambda_rat"] > 0}

_check_rows = []
for vname in VARIANTS:
    cfg = variant_cfg(vname)
    if cfg["tapt"] and TAPT_BACKBONE is None: continue           # not needed in this session
    if cfg["rationale"] and NOTE_EMB is None: continue
    m = build_model(cfg); m.train()
    b = make_collate(cfg, training_aux=True)([ScenarioDataset(train_df.head(2), cfg, augment=True, seed=1)[i] for i in range(2)])
    b.pop("ids"); b = {k: v.to(DEVICE) for k, v in b.items()}
    with amp_context():
        out = m(b["input_ids"], b["attention_mask"], policy_dropout=POLICY_DROPOUT); loss, pieces = total_loss(out, b, m, cfg)
    assert torch.isfinite(loss), vname
    for k, rule in EXPECTED_PIECES.items():
        assert (k in pieces) == rule(cfg), f"{vname}: loss term {k} presence wrong"
    assert (m.attn is not None) == cfg["attn_pool"]; assert (m.film is not None) == cfg["film"]
    assert m.dec_head[0].in_features == m.enc.config.hidden_size + (7 if cfg["intent"] else 0)
    _check_rows.append({"variant": vname, "backbone": "TAPT/TLM" if cfg["tapt"] else "plain",
                        "params_M": round(sum(p.numel() for p in m.parameters()) / 1e6, 2), "loss_terms": ",".join(pieces)})
    del m, b, out; free_memory()
display(pd.DataFrame(_check_rows))
print("Switch verification: PASS")


In [ ]:
# ============================================================
# CELL 11 — TRAIN / PREDICT / METRICS
# ============================================================
def optimizer_groups(model):
    n_layers = int(getattr(model.enc.config, "num_hidden_layers", 12)); groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad: continue
        wd = 0.0 if ("bias" in name or "LayerNorm" in name or "layer_norm" in name.lower()) else WEIGHT_DECAY
        if not name.startswith("enc."): lr = LR_HEADS
        else:
            mm = re.search(r"encoder\.layer\.(\d+)", name)
            if mm: lr = LR_ENCODER * (LAYERWISE_LR_DECAY ** (n_layers - 1 - int(mm.group(1))))
            elif "embeddings" in name: lr = LR_ENCODER * (LAYERWISE_LR_DECAY ** n_layers)
            else: lr = LR_ENCODER
        groups.setdefault((round(lr, 12), wd), []).append(p)
    return [{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in groups.items()]

@torch.no_grad()
def predict(model, df, cfg):
    model.eval(); recs = []; offset = 0
    dl = DataLoader(ScenarioDataset(df, cfg, augment=False), batch_size=EVAL_SCENARIO_BATCH, shuffle=False, collate_fn=make_collate(cfg, False))
    for batch in dl:
        ids = batch.pop("ids"); x = {k: batch[k].to(DEVICE) for k in ["input_ids", "attention_mask"]}
        with amp_context(): out = model(x["input_ids"], x["attention_mask"], policy_dropout=0.0)
        probs = torch.softmax(out["dec"].float(), -1).cpu().numpy(); ip = out["int"].float().argmax(-1).cpu().numpy()
        for bi in range(len(ids)):
            src = df.iloc[offset + bi]
            for vi, v in enumerate(TEXT_VIEWS):
                j = bi * 4 + vi
                recs.append({"id": ids[bi], "scenario_family_id": src.scenario_family_id, "domain": src.domain,
                             "security_intent": src.security_intent, "hard_negative_type": src.hard_negative_type,
                             "in_decision_matched_test": bool(src.in_decision_matched_test), "text_view": v,
                             "y_true": LABEL2ID[src.decision], "p_ALLOW": float(probs[j, 0]), "p_REVIEW": float(probs[j, 1]),
                             "p_BLOCK": float(probs[j, 2]), "pred_id": int(probs[j].argmax()),
                             "intent_true_id": INTENT2ID[src.security_intent], "intent_pred_id": int(ip[j])})
        offset += len(ids)
    return pd.DataFrame(recs)

def run_metrics(p, has_intent_head=True):
    y = p.y_true.values; pr = p.pred_id.values
    yb = (y > 0).astype(int); pb = (pr > 0).astype(int); score = 1 - p.p_ALLOW.values
    f1s = precision_recall_fscore_support(y, pr, labels=[0, 1, 2], zero_division=0)[2]
    hn = (p.hard_negative_type == "hard_negative").values; dm = p.in_decision_matched_test.values.astype(bool)
    out = {"macro_f1": f1_score(y, pr, average="macro", labels=[0, 1, 2], zero_division=0), "accuracy": accuracy_score(y, pr),
           "f1_ALLOW": f1s[0], "f1_REVIEW": f1s[1], "f1_BLOCK": f1s[2],
           "detection_f1": f1_score(yb, pb, average="macro", zero_division=0),
           "recall": float(pb[yb == 1].mean()), "specificity": float(1 - pb[yb == 0].mean()),
           "over_refusal": float(pb[hn].mean()) if hn.any() else np.nan,
           "matched_macro_f1": f1_score(y[dm], pr[dm], average="macro", labels=[0, 1, 2], zero_division=0) if dm.any() else np.nan,
           "intent_macro_f1": (f1_score(p.intent_true_id, p.intent_pred_id, average="macro", labels=list(range(7)), zero_division=0)
                               if has_intent_head else np.nan)}
    try: out["auroc"] = roc_auc_score(yb, score)
    except Exception: out["auroc"] = np.nan
    return out

def train_run(cfg, seed):
    model = build_model(cfg); seed_all(seed)
    gen = torch.Generator(); gen.manual_seed(seed)
    dl = DataLoader(ScenarioDataset(train_df, cfg, augment=True, seed=seed), batch_size=TRAIN_SCENARIO_BATCH, shuffle=True,
                    collate_fn=make_collate(cfg, training_aux=True), generator=gen)
    opt = torch.optim.AdamW(optimizer_groups(model))
    epochs = 1 if SMOKE_RUN else EPOCHS
    total = SMOKE_MAX_STEPS if SMOKE_RUN else max(1, math.ceil(len(dl) / GRAD_ACCUM)) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP_RATIO * total), total); scaler = make_scaler()
    best = -1; best_state = None; bad = 0; step = 0; hist = []; t0 = time.time()
    for ep in range(epochs):
        model.train(); opt.zero_grad(set_to_none=True); losses = []
        for bi, batch in enumerate(dl):
            batch.pop("ids"); batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with amp_context():
                out = model(batch["input_ids"], batch["attention_mask"], policy_dropout=POLICY_DROPOUT if cfg["film"] else 0.0)
                loss, _ = total_loss(out, batch, model, cfg); loss = loss / GRAD_ACCUM
            scaler.scale(loss).backward(); losses.append(float(loss.detach()) * GRAD_ACCUM)
            if (bi + 1) % GRAD_ACCUM == 0 or (bi + 1) == len(dl):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
                if SMOKE_RUN and step >= SMOKE_MAX_STEPS: break
        vp = predict(model, val_df, cfg)
        vf1 = f1_score(vp.y_true, vp.pred_id, average="macro", labels=[0, 1, 2], zero_division=0)
        hist.append({"epoch": ep + 1, "step": step, "train_loss": float(np.mean(losses)), "val_macro_f1": float(vf1),
                     "elapsed_min": (time.time() - t0) / 60})
        print("   ", hist[-1])
        if vf1 > best + 1e-3:
            best = vf1; bad = 0; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        if SMOKE_RUN: break
        if bad >= PATIENCE: print("    early stop"); break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist), best, (time.time() - t0) / 60


In [ ]:
# ============================================================
# CELL 12 — IMPORT FINISHED RUNS FROM ATTACHED OUTPUTS, THEN RUN THIS SESSION
# ============================================================
def import_previous_runs():
    n = 0; root = Path("/kaggle/input")
    if SMOKE_RUN or not root.exists(): return 0
    for done in root.rglob("ablation_runs/*/seed*/done.json"):
        dst = RUNS_DIR / done.parent.parent.name / done.parent.name
        if not (dst / "done.json").exists():
            shutil.copytree(done.parent, dst, dirs_exist_ok=True); n += 1
    return n

print("imported finished runs from attached inputs:", import_previous_runs())

for vname in RUN_VARIANTS:
    cfg = variant_cfg(vname)
    for seed in RUN_SEEDS:
        rdir = RUNS_DIR / vname / f"seed{seed}"
        if (rdir / "done.json").exists():
            print(f"[skip] {vname} seed {seed} (already finished)"); continue
        rdir.mkdir(parents=True, exist_ok=True)
        print(f"\n=========== {vname} | seed {seed} ===========")
        model, hist, best_val, train_min = train_run(cfg, seed)
        hist.to_csv(rdir / "history.csv", index=False)
        tp = predict(model, test_df, cfg); sp = predict(model, stress_df, cfg)
        tp.to_csv(rdir / "predictions_test.csv.gz", index=False); sp.to_csv(rdir / "predictions_stress.csv.gz", index=False)
        m = run_metrics(tp, cfg["intent"]); ms = run_metrics(sp, cfg["intent"])
        summary = {"variant": vname, "seed": seed, "val_macro_f1": best_val, "epochs_run": int(hist.epoch.max()),
                   "train_minutes": train_min, "params": int(sum(p.numel() for p in model.parameters())),
                   **{f"test_{k}": v for k, v in m.items()}, **{f"stress_{k}": v for k, v in ms.items()},
                   "config": {k: cfg[k] for k in BASE_CFG}, "smoke_run": SMOKE_RUN,
                   "backbone": backbone_for(cfg), "gpu": GPU_NAME}
        with open(rdir / "done.json", "w") as f: json.dump(summary, f, indent=2, default=float)
        print(f"    test 3-class F1 {m['macro_f1']:.4f} | detection F1 {m['detection_f1']:.4f} | "
              f"over-refusal {m['over_refusal']:.3f} | stress F1 {ms['macro_f1']:.4f} | {train_min:.1f} min")
        del model; free_memory()
print("\nsession finished")


In [ ]:
# ============================================================
# CELL 13 — SUMMARY: MEAN ± SD AND PAIRED FAMILY BOOTSTRAP VS FULL
#   Same bootstrap as the paper: family-clustered, 5,000 resamples, seed-averaged.
# ============================================================
def load_runs():
    rows, preds = [], {}
    for done in sorted(RUNS_DIR.glob("*/seed*/done.json")):
        s = json.load(open(done)); rows.append(s)
        preds[(s["variant"], s["seed"])] = {"test": pd.read_csv(done.parent / "predictions_test.csv.gz"),
                                            "stress": pd.read_csv(done.parent / "predictions_stress.csv.gz")}
    return pd.DataFrame(rows), preds

RUNS, PREDS = load_runs()
print("finished runs:", len(RUNS))
KEYS = ["test_macro_f1", "test_f1_REVIEW", "test_detection_f1", "test_recall", "test_over_refusal",
        "test_matched_macro_f1", "test_intent_macro_f1", "stress_macro_f1", "epochs_run", "train_minutes"]
order = [v for v in VARIANTS if v in set(RUNS.variant)] if len(RUNS) else []
if len(RUNS):
    g = RUNS.groupby("variant")[KEYS]; summ = g.mean().loc[order]; sd = g.std(ddof=1).loc[order]; n = RUNS.groupby("variant").size().loc[order]
    fmt = lambda m, s: "—" if pd.isna(m) else (f"{m:.4f}" if pd.isna(s) else f"{m:.4f} ± {s:.4f}")
    table = pd.DataFrame({c: [fmt(summ.loc[v, c], sd.loc[v, c]) for v in order] for c in KEYS}, index=order); table.insert(0, "n_seeds", n.values)
    display(table); summ.to_csv(OUT_DIR / "ablation_mean.csv"); sd.to_csv(OUT_DIR / "ablation_sd.csv")

def _counts(d, kind, fidx, F):
    fi = d.scenario_family_id.map(fidx).values
    if kind == "3":
        C = np.zeros((F, 9)); np.add.at(C, (fi, d.y_true.values * 3 + d.pred_id.values), 1)
    elif kind == "bin":
        C = np.zeros((F, 4)); np.add.at(C, (fi, (d.y_true.values > 0) * 2 + (d.pred_id.values > 0)), 1)
    else:
        h = (d.hard_negative_type == "hard_negative").values; C = np.zeros((F, 2))
        np.add.at(C, (fi[h], 0), 1); np.add.at(C, (fi[h], 1), (d.pred_id.values[h] > 0))
    return C

def _score(CM, kind):
    if kind == "or": return CM[:, 1] / np.maximum(CM[:, 0], 1)
    k = 3 if kind == "3" else 2; M = CM.reshape(-1, k, k)
    tp = np.diagonal(M, axis1=1, axis2=2); fp = M.sum(1) - tp; fn = M.sum(2) - tp
    return (2 * tp / np.maximum(2 * tp + fp + fn, 1e-9)).mean(1)

pair_rows = []
if len(RUNS) and "full" in set(RUNS.variant):
    fams = np.array(sorted(test_df.scenario_family_id.unique())); fidx = {f: i for i, f in enumerate(fams)}; F_ = len(fams)
    W = np.random.default_rng(2026).multinomial(F_, np.ones(F_) / F_, size=N_BOOT).astype(float)
    def boot(v, split, kind):
        seeds = sorted(RUNS[RUNS.variant == v].seed); cs = [_counts(PREDS[(v, s)][split], kind, fidx, F_) for s in seeds]
        return np.mean([_score(W @ C, kind) for C in cs], 0), np.mean([_score(C.sum(0, keepdims=True), kind)[0] for C in cs])
    for v in order:
        if v == "full": continue
        for split, kind, label in [("test", "3", "three_class_f1"), ("test", "bin", "detection_f1"),
                                   ("test", "or", "over_refusal"), ("stress", "3", "stress_three_class_f1")]:
            bv, pv = boot(v, split, kind); bf, pf = boot("full", split, kind); d = bv - bf; pt = pv - pf
            p1 = (d >= 0).mean() if pt < 0 else (d <= 0).mean()
            pair_rows.append({"variant": v, "metric": label, "delta_vs_full_pp": 100 * pt,
                              "ci_lo_pp": 100 * np.percentile(d, 2.5), "ci_hi_pp": 100 * np.percentile(d, 97.5),
                              "p_two_sided": min(1.0, 2 * p1)})
    PAIR = pd.DataFrame(pair_rows); PAIR.to_csv(OUT_DIR / "ablation_paired_vs_full.csv", index=False)
    display(PAIR.pivot(index="variant", columns="metric", values="delta_vs_full_pp").loc[[v for v in order if v != "full"]].round(2))
else:
    print("paired bootstrap needs the 'full' variant (group 1)")


In [ ]:
# ============================================================
# CELL 14 — PACKAGE RESULTS FOR DOWNLOAD
# ============================================================
zip_base = WORK_ROOT / ("banglablocker_ablation_results" + ("_smoke" if SMOKE_RUN else ""))
keep = OUT_DIR / "_package"; shutil.rmtree(keep, ignore_errors=True); keep.mkdir(parents=True)
shutil.copytree(RUNS_DIR, keep / "ablation_runs")
for f in OUT_DIR.glob("ablation_*.csv"): shutil.copy2(f, keep / f.name)
json.dump({"variants": {v: variant_cfg(v) for v in VARIANTS}, "groups": GROUPS, "seeds": SEEDS, "smoke_run": SMOKE_RUN,
           "hyperparameters": {"MAX_LEN": MAX_LEN, "EPOCHS": EPOCHS, "PATIENCE": PATIENCE, "LR_ENCODER": LR_ENCODER,
                               "LR_HEADS": LR_HEADS, "LLRD": LAYERWISE_LR_DECAY, "batch_scenarios": TRAIN_SCENARIO_BATCH * GRAD_ACCUM,
                               "TAPT_EPOCHS": TAPT_EPOCHS}, "gpu": GPU_NAME, "torch": torch.__version__},
          open(keep / "ablation_config.json", "w"), indent=2, default=str)
path = shutil.make_archive(str(zip_base), "zip", root_dir=keep)
shutil.rmtree(keep, ignore_errors=True)
print("download:", path, f"({os.path.getsize(path) / 1e6:.1f} MB)")
